In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"


import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images
from mapanything.utils.image import preprocess_inputs
import open3d as o3d
import numpy as np
from PIL import Image
import copy
import gtsam

# --- Configuration ---
device = "cuda" if torch.cuda.is_available() else "cpu"
BASE_IMAGE_PATH = "/home/tong/recordings/scripps924/scripps924_5/"
MAX_FILTER_DISTANCE = 15.0 
VOXEL_SIZE = 0.1 # 10cm voxel size for fusing.


KEYFRAME_STEP = 60       # The gap between keyframes (e.g., 10, 20, 30)
END_FRAME = 1300         # The last frame number you want to process
NUM_VIEWS_PER_BATCH = 5  
# ----------------------------------------

# --- GTSAM CONFIGURATION  ---
ODOMETRY_NOISE = gtsam.noiseModel.Diagonal.Sigmas(
    np.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1])
)

LOOP_NOISE = gtsam.noiseModel.Diagonal.Sigmas(
    np.array([0.01, 0.01, 0.01, 0.01, 0.01, 0.01])
)

PRIOR_NOISE = gtsam.noiseModel.Diagonal.Sigmas(
    np.array([1e-6, 1e-6, 1e-6, 1e-6, 1e-6, 1e-6])
)

# How far (in meters) to search for old nodes to link to
LOCAL_SEARCH_RADIUS = VOXEL_SIZE * 10 # e.g. 0.1 * 10 = 1.0 meter
LOCAL_WINDOW_SIZE = 5

# --- Model and Intrinsics ---
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

transform_matrix = np.array([
    [1,  0,  0,  0],  # Stays the same
    [0, -1,  0,  0],  # Inverts Y
    [0,  0, -1,  0],  
    [0,  0,  0,  1]
])

intrinsics = np.array([
    [987.46, 0.0, 830.36],
    [0.0, 987.46, 644.75],
    [0.0,    0.0,   1.0 ],
], dtype=np.float32)

# --- Global Lists ---
geometries = [] # Stores final *visualization* geometries
poses = []      # Stores final *optimized* global poses (as np.array)
global_pcd_list = [] # Stores *raw* global point clouds (for ICP)

# --- NEW: Global Graph Objects ---
graph = gtsam.NonlinearFactorGraph()
initial_estimates = gtsam.Values()
pcd_database = {}

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /home/tong/.cache/torch/hub/facebookresearch_dinov2_main


In [2]:
def np_to_gtsam(pose_np):
    """Convert numpy 4x4 matrix to gtsam.Pose3"""
    rot_np = pose_np[0:3, 0:3]
    trans_np = pose_np[0:3, 3]
    return gtsam.Pose3(gtsam.Rot3(rot_np), gtsam.Point3(trans_np))

def gtsam_to_np(pose_gtsam):
    """Convert gtsam.Pose3 to numpy 4x4 matrix"""
    pose_np = np.eye(4)
    pose_np[0:3, 0:3] = pose_gtsam.rotation().matrix()
    pose_np[0:3, 3] = pose_gtsam.translation()
    return pose_np.astype(np.float32)

def run_icp(source_pcd, target_pcd_fused, initial_guess_transform):
    """Helper to run ICP and return the result"""
    # We need a source cloud that is *already moved* by the VO guess
    source_cloud_transformed = copy.deepcopy(source_pcd)
    source_cloud_transformed.transform(initial_guess_transform)
    
    # ICP needs normals for point-to-plane
    target_pcd_fused.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=VOXEL_SIZE * 2, max_nn=30))
    source_cloud_transformed.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=VOXEL_SIZE * 2, max_nn=30))
    
    # Run ICP
    icp_result = o3d.pipelines.registration.registration_icp(
        source_cloud_transformed, 
        target_pcd_fused, 
        VOXEL_SIZE * 2, # Use voxel size * 2 as correspondence threshold
        np.identity(4),  # We use identity matrix because we already transformed the source
        o3d.pipelines.registration.TransformationEstimationPointToPlane()
    )
    return icp_result

In [ ]:
def initial_map(preds):
    """Processes the very first batch AND initializes the GTSAM graph."""
    global graph, initial_estimates, poses, pcd_database

    print("--- Initializing Map and Graph ---")

    # --- NEW: Add the "anchor" to the graph ---
    # We add a "prior" to the first pose (Node 0) to fix it at the origin.
    # This "nails" the map in place.
    first_pose_gtsam = gtsam.Pose3() # This is an identity matrix
    graph.add(gtsam.PriorFactorPose3(0, first_pose_gtsam, PRIOR_NOISE))
    initial_estimates.insert(0, first_pose_gtsam)
    
    # Also add it to our Python lists
    first_pose_np = gtsam_to_np(first_pose_gtsam)
    poses.append(first_pose_np)
    
    last_pose_np = first_pose_np

    for i, pred in enumerate(preds):
        # Get raw points, colors, and the pose
        points_world = pred["pts3d"].squeeze().cpu().numpy().reshape(-1, 3)
        colors = pred["img_no_norm"].squeeze().cpu().numpy().reshape(-1, 3)
        camera_pose_np = pred["camera_poses"].squeeze().cpu().numpy().astype(np.float32)

        # --- Filtering---
        camera_origin = camera_pose_np[:3, 3]
        distances = np.linalg.norm(points_world - camera_origin, axis=1)
        mask = distances <= MAX_FILTER_DISTANCE
        filtered_points = points_world[mask]
        filtered_colors = colors[mask]
        
        # Create raw point cloud for ICP and storage
        pcd_raw = o3d.geometry.PointCloud()
        pcd_raw.points = o3d.utility.Vector3dVector(filtered_points)
        pcd_raw.colors = o3d.utility.Vector3dVector(filtered_colors)
        
        # Store in our lists
        global_pcd_list.append(pcd_raw)
        pcd_database[i] = pcd_raw # Store in the full database
        
        # --- NEW: Add nodes and odometry edges to the graph ---
        if i > 0: # We already added Node 0
            # Add the new node
            pose_gtsam = np_to_gtsam(camera_pose_np)
            initial_estimates.insert(i, pose_gtsam)
            poses.append(camera_pose_np) # Add to our numpy list
            
            # Add the odometry constraint (edge)
            relative_pose = np_to_gtsam(np.linalg.inv(last_pose_np) @ camera_pose_np)
            graph.add(gtsam.BetweenFactorPose3(i-1, i, relative_pose, ODOMETRY_NOISE))
            
        last_pose_np = camera_pose_np # Update for next loop

        # --- Visualization (no change) ---
        pcd_vis = copy.deepcopy(pcd_raw)
        pcd_vis.transform(transform_matrix)
        
        camera_frame_vis = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5) 
        camera_frame_vis.transform(camera_pose_np)
        camera_frame_vis.transform(transform_matrix)
        
        geometries.append(pcd_vis)
        geometries.append(camera_frame_vis)

    # --- NEW: Run an initial optimization ---
    print("Running initial graph optimization...")
    optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial_estimates)
    result = optimizer.optimize()
    
    # Update all our poses with the optimized results
    initial_estimates = result
    for i in range(len(poses)):
        poses[i] = gtsam_to_np(result.atPose3(i))
    
    print(f"Initial map created. Graph has {graph.size()} factors.")

In [ ]:
def view_process(batch):
    """Prepares a batch of images for the model."""
    views = []
    for i, img in enumerate(batch):
        if i == len(batch)-1:
            # This is the new frame
            views.append({
                "img": np.array(Image.open(img).convert("RGB")),
                "intrinsics": intrinsics,
                "is_metric_scale": torch.tensor([True], device=device),
            })
        else:
            # These are "anchor" frames. We give their *most recent optimized poses*.
            anchor_pose_index = len(poses) - 4 + i
            views.append({
                "img": np.array(Image.open(img).convert("RGB")),
                "intrinsics": intrinsics,
                "camera_poses": poses[anchor_pose_index], 
                "is_metric_scale": torch.tensor([True], device=device),
            })
    processed_views = preprocess_inputs(views)

    return processed_views